# I/Q Data (IQ Sampling) — Interactive Lab
Based on *"I/Q Data for Dummies"* (whiteboard.ping.se/SDR/IQ).

**How to use:** run cells top to bottom. Every section has **sliders**; sections with a ▶ **Play** button animate time.

| # | Section |
|---|---------|
| 1 | Why I/Q? (sign ambiguity, amplitude ambiguity) |
| 2 | The corkscrew: side, top, and down-the-time-axis views |
| 3 | One I/Q sample (phasor) |
| 4 | Rectangular, polar, Euler forms |
| 5 | Positive vs negative frequency |
| 6 | Nyquist: real vs I/Q bandwidth |
| 7 | Mixing: real vs I/Q (spectrum) |
| 8 | Mixing *f* with *−f* (time domain) |
| 9 | RF carrier: modulate and down-convert |
| 10 | AM and FM in I/Q |
| 11 | "A little trickery": frequency flips sign |
| 12 | Pendulum analogy |

Requires: `numpy`, `matplotlib`, `ipywidgets` (`pip install ipywidgets`).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

_STYLE = {"description_width": "initial"}

def S(desc, lo, hi, step, val, live=True, width="340px"):
    # float slider helper
    return w.FloatSlider(description=desc, min=lo, max=hi, step=step, value=val,
                         continuous_update=live, readout_format=".2f",
                         style=_STYLE, layout=w.Layout(width=width))

def player(n, step=1, interval=40, desc="time index"):
    # slider + Play button linked together (animation)
    sl = w.IntSlider(description=desc, min=0, max=n - 1, step=1, value=0,
                     continuous_update=True, style=_STYLE, layout=w.Layout(width="340px"))
    pl = w.Play(min=0, max=n - 1, step=step, value=0, interval=interval)
    w.jslink((pl, "value"), (sl, "value"))
    return sl, pl

def ui(func, extras=(), **ctrls):
    # show controls + live plot output
    out = w.interactive_output(func, ctrls)
    box = w.HBox(list(extras) + list(ctrls.values()), layout=w.Layout(flex_flow="row wrap"))
    display(box, out)

def helix_figure(t, s, k, elev=20, azim=-60):
    # 3D corkscrew + side view (I) + top view (Q) + view down the time axis
    I, Q = s.real, s.imag
    m = 1.3 * np.max(np.abs(s))
    fig = plt.figure(figsize=(11, 6.5))
    gs = fig.add_gridspec(3, 2, width_ratios=[1.3, 1])
    ax = fig.add_subplot(gs[:, 0], projection="3d")
    ax.plot(t, I, Q, lw=1.2, color="C0")
    ax.plot(t, I, np.full_like(t, -m), color="C1", alpha=0.5, lw=1)   # side view  -> I(t)
    ax.plot(t, np.full_like(t, -m), Q, color="C2", alpha=0.5, lw=1)   # top view   -> Q(t)
    ax.plot([t[k], t[k]], [0, I[k]], [0, Q[k]], "r-", lw=2)
    ax.plot([t[k]], [I[k]], [Q[k]], "ro")
    ax.set_xlabel("time"); ax.set_ylabel("I"); ax.set_zlabel("Q")
    ax.set_xlim(t[0], t[-1]); ax.set_ylim(-m, m); ax.set_zlim(-m, m)
    ax.view_init(elev=elev, azim=azim)
    ax.set_title("Corkscrew: (time, I, Q)")

    a1 = fig.add_subplot(gs[0, 1])
    a1.plot(t, I, "C1"); a1.plot(t[k], I[k], "ro")
    a1.set_ylim(-m, m); a1.set_title("Side view = I (your 'real' signal)")

    a2 = fig.add_subplot(gs[1, 1])
    a2.plot(t, Q, "C2"); a2.plot(t[k], Q[k], "ro")
    a2.set_ylim(-m, m); a2.set_title("Top view = Q")

    a3 = fig.add_subplot(gs[2, 1])
    a3.plot(I, Q, color="0.8")
    a3.plot(I[:k + 1], Q[:k + 1], "C0")
    a3.annotate("", xy=(I[k], Q[k]), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="r", lw=2))
    a3.set_xlim(-m, m); a3.set_ylim(-m, m); a3.set_aspect("equal")
    a3.set_xlabel("I"); a3.set_ylabel("Q")
    a3.set_title(f"Down the time axis: A={abs(s[k]):.2f}, phase={np.degrees(np.angle(s[k])):.0f}°")
    fig.tight_layout()
    plt.show()

## 1. Why I/Q?
**Problem 1 – sign of frequency.** $\cos(x)=\cos(-x)$, so a real signal can't tell $+f$ from $-f$.
**Problem 2 – amplitude.** From real samples you only see the peak if you happen to sample at it.

Move **f** (the two curves always coincide) and **sampling offset** (the real samples change, the I/Q amplitude $\sqrt{I^2+Q^2}$ stays constant).

In [2]:
def sec1(f, offset):
    t = np.linspace(0, 2, 1000)
    ts = offset + np.arange(0, 2 - 0.125, 0.125)           # 8 samples per unit time
    I = np.cos(2 * np.pi * f * ts)
    Q = np.sin(2 * np.pi * f * ts)
    A = np.hypot(I, Q)

    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))
    a.plot(t, np.cos(2 * np.pi * f * t), label=f"cos, f = {f:+.2f}")
    a.plot(t, np.cos(2 * np.pi * (-f) * t), "--", label=f"cos, f = {-f:+.2f}")
    a.stem(ts, I, linefmt="r-", markerfmt="ro", basefmt=" ")
    a.set_title("Real signal: +f and -f are identical")
    a.legend(loc="upper right"); a.set_ylim(-1.4, 1.4)

    b.stem(ts - 0.005, np.abs(I), linefmt="r-", markerfmt="ro", basefmt=" ", label="|I| (real samples)")
    b.stem(ts + 0.005, A, linefmt="g-", markerfmt="gs", basefmt=" ", label="A = sqrt(I²+Q²)")
    b.set_ylim(0, 1.4); b.set_title("Amplitude estimate per sample")
    b.legend(loc="upper right")
    fig.tight_layout(); plt.show()

ui(sec1, f=S("frequency", -3, 3, 0.25, 1.0), offset=S("sampling offset", 0, 0.125, 0.005, 0.0))

Output()

## 2. The corkscrew (helix)
$s(t)=e^{j2\pi f t}=\cos(2\pi f t)+j\sin(2\pi f t)$

- **Side view** → $I$ (the ordinary real signal)
- **Top view** → $Q$ (same wave, 90° shifted)
- **Down the time axis** → circle; radius = amplitude; direction of winding = sign of frequency

▶ Press **Play**, drag **elev/azim** to rotate the 3D view, and set **frequency < 0** to reverse the winding.

In [3]:
T2 = np.linspace(0, 2, 400)

def sec2(f, elev, azim, k):
    helix_figure(T2, np.exp(2j * np.pi * f * T2), k, elev, azim)

k2, p2 = player(len(T2), step=2)
ui(sec2, extras=[p2],
   f=S("frequency (neg = clockwise)", -3, 3, 0.25, 1.0, live=False),
   elev=S("elev", 0, 90, 5, 20, live=False), azim=S("azim", -180, 180, 5, -60, live=False),
   k=k2)

Output()

## 3. One I/Q sample (phasor)
Example from the page: $I=0.69,\ Q=0.40 \Rightarrow A=0.8,\ \phi=30^\circ$, so the real signal is $0.8\cos(30^\circ)=0.69$.

Move **I** and **Q**: the point on the cosine wave (right) is where this sample sits in its cycle.

In [7]:
def sec3(I, Q):
    A = np.hypot(I, Q)
    phi = np.degrees(np.arctan2(Q, I))
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
    th = np.linspace(0, 2 * np.pi, 200)
    a.plot(A * np.cos(th), A * np.sin(th), color="0.8")
    a.annotate("", xy=(I, Q), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="r", lw=2))
    a.plot([I, I], [0, Q], "b--"); a.plot([0, I], [Q, Q], "g--")
    a.set_xlim(-1.6, 1.6); a.set_ylim(-1.6, 1.6); a.set_aspect("equal")
    a.set_xlabel("I (real)"); a.set_ylabel("Q (imag)")
    a.set_title(f"Phasor: I={I:.2f}, Q={Q:.2f}")

    x = np.linspace(0, 360, 500)
    b.plot(x, A * np.cos(np.radians(x)), label="A·cos(φ)")
    b.plot([phi % 360], [I], "ro", ms=9)
    b.axhline(I, color="r", ls=":")
    b.set_xlabel("angle φ (degrees)"); b.set_ylim(-1.6, 1.6)
    b.set_title(f"A = {A:.2f},  φ = {phi:.1f}°,  I = A·cos(φ) = {I:.2f}")
    b.legend(loc="lower left")
    fig.tight_layout(); plt.show()

ui(sec3, I=S("I", -1, 1, 0.01, 0.69), Q=S("Q", -1, 1, 0.01, 0.40))

Output()

## 4. Three ways to write the same sample
| Form | Expression |
|---|---|
| Rectangular | $I + jQ$ |
| Polar | $A\angle\phi$, with $I=A\cos\phi,\ Q=A\sin\phi$ |
| Euler | $A e^{j\phi}$ |

Formulas: $A=\sqrt{I^2+Q^2}$, $\phi=\tan^{-1}(Q/I)$.
Set **φ = 180°** → $e^{j\pi}=-1$ (Euler's identity $e^{j\pi}+1=0$). ▶ Play rotates φ.

In [8]:
def sec4(A, phi):
    z = A * np.exp(1j * np.radians(phi))
    print(f"Rectangular : I = {z.real:+.3f},  Q = {z.imag:+.3f}   ->  {z.real:+.3f} {z.imag:+.3f}j")
    print(f"Polar       : A = {A:.3f},  phi = {phi:.1f} deg")
    print(f"Euler       : {A:.2f} * exp(j*{np.radians(phi):.3f})")
    print(f"Check       : sqrt(I²+Q²) = {np.hypot(z.real, z.imag):.3f},  atan2(Q,I) = {np.degrees(np.arctan2(z.imag, z.real)):.1f} deg")
    print(f"exp(j*pi)+1 = {abs(np.exp(1j*np.pi) + 1):.1e}  (≈ 0)")

    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(A * np.cos(th), A * np.sin(th), color="0.8")
    ax.annotate("", xy=(z.real, z.imag), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="r", lw=2))
    ax.plot([z.real, z.real], [0, z.imag], "b--", label="Q (projection)")
    ax.plot([0, z.real], [0, 0], "g-", lw=3, label="I (projection)")
    arc = np.linspace(0, np.radians(phi), 60)
    ax.plot(0.25 * np.cos(arc), 0.25 * np.sin(arc), "k")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6); ax.set_aspect("equal")
    ax.set_xlabel("I (real)"); ax.set_ylabel("Q (imag)"); ax.legend(loc="lower right")
    plt.show()

phi_sl = S("phase φ (deg)", 0, 360, 5, 30)
play4 = w.Play(min=0, max=360, step=5, value=30, interval=60)
w.jslink((play4, "value"), (phi_sl, "value"))
ui(sec4, extras=[play4], A=S("amplitude A", 0.1, 1.5, 0.05, 0.8), phi=phi_sl)

Output(outputs=({'name': 'stdout', 'text': 'Rectangular : I = +0.693,  Q = +0.400   ->  +0.693 +0.400j\nPolar …

## 5. Positive vs negative frequency
Blue = $+f$, red = $-f$ (red slightly phase-shifted so the real parts can be told apart).
Compare the **I** panels (almost the same) with the **Q** panels (opposite sign) and the phasor (opposite rotation).

In [9]:
T5 = np.linspace(0, 2, 400)

def sec5(f, dphi, k):
    t = T5
    sb = np.exp(2j * np.pi * f * t)                               # +f
    sr = np.exp(-2j * np.pi * f * t + 1j * np.radians(dphi))      # -f, phase shifted
    fig = plt.figure(figsize=(11, 4.8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.4])
    p = fig.add_subplot(gs[:, 0])
    for s, c, n in ((sb, "C0", "+f"), (sr, "C3", "-f")):
        p.plot(s.real, s.imag, color=c, alpha=0.25)
        p.plot(s.real[max(0, k - 40):k + 1], s.imag[max(0, k - 40):k + 1], color=c, lw=2)
        p.annotate("", xy=(s[k].real, s[k].imag), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=c, lw=2))
    p.set_xlim(-1.3, 1.3); p.set_ylim(-1.3, 1.3); p.set_aspect("equal")
    p.set_xlabel("I"); p.set_ylabel("Q"); p.set_title("Phasors (blue ccw, red cw)")
    a = fig.add_subplot(gs[0, 1]); b = fig.add_subplot(gs[1, 1])
    a.plot(t, sb.real, "C0"); a.plot(t, sr.real, "C3"); a.plot(t[k], sb.real[k], "o", color="C0"); a.plot(t[k], sr.real[k], "o", color="C3")
    a.set_title("I (real part): hard to tell apart")
    b.plot(t, sb.imag, "C0"); b.plot(t, sr.imag, "C3"); b.plot(t[k], sb.imag[k], "o", color="C0"); b.plot(t[k], sr.imag[k], "o", color="C3")
    b.set_title("Q: opposite sign reveals the frequency sign"); b.set_xlabel("time")
    fig.tight_layout(); plt.show()

k5, p5 = player(len(T5), step=2)
ui(sec5, extras=[p5], f=S("frequency", 0.5, 3, 0.25, 1.0), dphi=S("phase shift of red (deg)", 0, 90, 5, 20), k=k5)

Output()

## 6. Nyquist: I/Q doubles the usable range
Sampling rate $f_s=100$. A real signal covers $[0, f_s/2]$ (spectrum is symmetric, $\pm f$ both appear).
I/Q covers $[-f_s/2, +f_s/2)$ with **one** peak whose sign is known. Push **f** past ±50 to see aliasing.

In [10]:
def sec6(f):
    fs, N = 100.0, 200
    n = np.arange(N)
    xr = np.cos(2 * np.pi * f * n / fs)
    xc = np.exp(2j * np.pi * f * n / fs)
    fr = np.fft.fftshift(np.fft.fftfreq(N, 1 / fs))
    Xr = np.abs(np.fft.fftshift(np.fft.fft(xr))) / N
    Xc = np.abs(np.fft.fftshift(np.fft.fft(xc))) / N
    alias = (f + fs / 2) % fs - fs / 2

    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    for ax, X, ttl in ((a, Xr, "Real signal: two mirror peaks"), (b, Xc, "I/Q signal: one peak, sign known")):
        ax.plot(fr, X)
        ax.axvspan(-fs / 2, fs / 2, color="g", alpha=0.05)
        ax.set_xlim(-fs / 2, fs / 2); ax.set_ylim(0, 1.1)
        ax.set_xlabel("frequency"); ax.set_title(ttl)
    a.axvline(0, color="k", lw=0.5)
    fig.suptitle(f"true f = {f:+.0f}   ->   apparent f = {alias:+.0f}  (fs = {fs:.0f})")
    fig.tight_layout(); plt.show()

ui(sec6, f=S("signal frequency", -60, 60, 1, 15))

Output()

## 7. Mixing (multiplying) two signals
- **Real:** $\cos(2\pi f_1 t)\cdot\cos(2\pi f_2 t)$ → **both** $f_1+f_2$ and $f_1-f_2$ (and mirrors).
- **I/Q:** $e^{j2\pi f_1 t}\cdot e^{j2\pi f_2 t}=e^{j2\pi(f_1+f_2)t}$ → **one** result.

Units: kHz and ms ($f_s=100$ kHz). Page example: $f_1=10,\ f_2=3$.

In [11]:
def sec7(f1, f2):
    fs, N = 100.0, 1000
    t = np.arange(N) / fs                                   # ms
    real_mix = np.cos(2 * np.pi * f1 * t) * np.cos(2 * np.pi * f2 * t)
    iq_mix = np.exp(2j * np.pi * f1 * t) * np.exp(2j * np.pi * f2 * t)
    fr = np.fft.fftshift(np.fft.fftfreq(N, 1 / fs))
    Xr = np.abs(np.fft.fftshift(np.fft.fft(real_mix))) / N
    Xc = np.abs(np.fft.fftshift(np.fft.fft(iq_mix))) / N

    fig, ax = plt.subplots(3, 1, figsize=(10, 8))
    m = t < 1.0
    ax[0].plot(t[m], real_mix[m], "k", label="real mix (composite)")
    ax[0].plot(t[m], iq_mix.real[m], "g", label="I/Q mix, I part (single tone)")
    ax[0].set_xlabel("time (ms)"); ax[0].legend(loc="upper right"); ax[0].set_title("Time domain")
    ax[1].plot(fr, Xr, "k"); ax[1].set_title(f"Real mix spectrum: peaks at ±{f1+f2:g} and ±{abs(f1-f2):g} kHz")
    ax[2].plot(fr, Xc, "g"); ax[2].set_title(f"I/Q mix spectrum: single peak at {f1+f2:+g} kHz")
    for a in ax[1:]:
        a.set_xlim(-50, 50); a.set_xlabel("frequency (kHz)")
    fig.tight_layout(); plt.show()

ui(sec7, f1=S("f1 (kHz)", -20, 20, 1, 10), f2=S("f2 (kHz)", -20, 20, 1, 3))

Output()

## 8. Mixing $f$ with $-f$ (time domain)
- **Green (I/Q product):** frequency **0** (constant) but still **full amplitude** — a DC value is fine in I/Q.
- **Black (real product):** $\cos(\omega t+a)\cos(-\omega t+b)=\tfrac12[\cos(a-b)+\cos(2\omega t+a+b)]$ → a $2f$ tone on top of a DC offset (DC is often removed by AC coupling).

Phases **a, b** are the slight offsets mentioned on the page.

In [12]:
def sec8(f, a_deg, b_deg):
    t = np.linspace(0, 1, 1000)
    a, b = np.radians(a_deg), np.radians(b_deg)
    sA = np.exp(1j * (2 * np.pi * f * t + a))      # +f
    sB = np.exp(1j * (-2 * np.pi * f * t + b))     # -f
    prod = sA * sB                                  # complex product
    real = np.cos(2 * np.pi * f * t + a) * np.cos(-2 * np.pi * f * t + b)

    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(t, sA.real, "C0", label="+f (I)"); ax[0].plot(t, sB.real, "C3", label="-f (I)")
    ax[0].legend(loc="upper right"); ax[0].set_title("Inputs (real parts)")
    ax[1].plot(t, prod.real, "g", label="I of product"); ax[1].plot(t, prod.imag, "g--", label="Q of product")
    ax[1].set_ylim(-1.3, 1.3); ax[1].legend(loc="upper right")
    ax[1].set_title(f"I/Q product: constant, |z| = {abs(prod).mean():.2f}, frequency 0")
    ax[2].plot(t, real, "k"); ax[2].axhline(real.mean(), color="r", ls="--", label=f"DC = {real.mean():.2f}")
    ax[2].legend(loc="upper right"); ax[2].set_xlabel("time")
    ax[2].set_title(f"Real product: tone at 2f = {2*f:g}")
    fig.tight_layout(); plt.show()

ui(sec8, f=S("f", 1, 5, 1, 2), a_deg=S("phase a (deg)", 0, 90, 5, 10), b_deg=S("phase b (deg)", 0, 90, 5, 25))

Output()

## 9. RF carrier: encode I/Q, then down-convert
**Modulate:** $RF = I\cos(2\pi f_c t)+Q\sin(2\pi f_c t)$
**Demodulate:** $I=\mathrm{lowpass}(2\,RF\cos(2\pi f_c t)),\quad Q=\mathrm{lowpass}(2\,RF\sin(2\pi f_c t))$ (the factor 2 restores amplitude)

Baseband used: $I=a_I\cos(2\pi f_b t)$, $Q=a_Q\sin(2\pi f_b t)$.
**Oscillator phase error θ:** the recovered I/Q are rotated by θ but amplitude and relative angle are kept.

In [13]:
def lowpass(x, fs, cutoff):
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), 1 / fs)
    X[f > cutoff] = 0
    return np.fft.irfft(X, len(x))

def sec9(fb, aI, aQ, theta):
    fs, fc = 4000.0, 100.0
    t = np.arange(0, 1, 1 / fs)
    I = aI * np.cos(2 * np.pi * fb * t)
    Q = aQ * np.sin(2 * np.pi * fb * t)
    RF = I * np.cos(2 * np.pi * fc * t) + Q * np.sin(2 * np.pi * fc * t)
    th = np.radians(theta)
    Ir = lowpass(2 * RF * np.cos(2 * np.pi * fc * t - th), fs, 20)
    Qr = lowpass(2 * RF * np.sin(2 * np.pi * fc * t - th), fs, 20)

    fig = plt.figure(figsize=(11, 6))
    gs = fig.add_gridspec(2, 2)
    a = fig.add_subplot(gs[0, 0]); b = fig.add_subplot(gs[1, 0]); c = fig.add_subplot(gs[0, 1]); d = fig.add_subplot(gs[1, 1])
    m = t < 0.5
    a.plot(t[m], RF[m], lw=0.6); a.set_title(f"RF signal (carrier {fc:g} Hz)"); a.set_xlabel("time (s)")
    b.plot(t, I, label="I sent"); b.plot(t, Ir, "--", label="I recovered")
    b.plot(t, Q, label="Q sent"); b.plot(t, Qr, "--", label="Q recovered")
    b.legend(ncol=2, loc="upper right", fontsize=8); b.set_title("Baseband before / after"); b.set_xlabel("time (s)")
    c.plot(I, Q, label="sent"); c.plot(Ir, Qr, "--", label="recovered")
    c.set_aspect("equal"); c.set_xlim(-1.3, 1.3); c.set_ylim(-1.3, 1.3); c.legend(loc="upper right", fontsize=8)
    c.set_xlabel("I"); c.set_ylabel("Q"); c.set_title(f"I/Q plane (rotated by {-theta:.0f}° when θ≠0)")
    A0, A1 = np.hypot(I, Q), np.hypot(Ir, Qr)
    d.plot(t, A0, label="|IQ| sent"); d.plot(t, A1, "--", label="|IQ| recovered")
    d.set_ylim(0, 1.5); d.legend(loc="upper right", fontsize=8); d.set_title("Amplitude is unaffected by θ"); d.set_xlabel("time (s)")
    fig.tight_layout(); plt.show()

ui(sec9, fb=S("baseband f_b (Hz)", 1, 5, 1, 2), aI=S("a_I", 0, 1, 0.05, 0.8), aQ=S("a_Q", 0, 1, 0.05, 0.5),
   theta=S("LO phase error θ (deg)", 0, 180, 5, 0))

Output()

## 10. AM and FM in I/Q
- **AM:** $s(t)=\frac{1+m\cos(2\pi f_m t)}{1+m}e^{j2\pi f_c t}$ → radius changes, even when $I=0$ the amplitude $\sqrt{I^2+Q^2}$ is known.
- **FM:** $s(t)=e^{j(2\pi f_c t+\beta\sin(2\pi f_m t))}$ → radius constant, winding speed changes.

▶ Play, change **mode**, **m / β**, and rotate the view.

In [14]:
T10 = np.linspace(0, 4, 600)

def sec10(mode, m, beta, elev, azim, k):
    fc, fm = 3.0, 0.5
    t = T10
    if mode == "AM":
        s = (1 + m * np.cos(2 * np.pi * fm * t)) / (1 + m) * np.exp(2j * np.pi * fc * t)
    else:
        s = np.exp(1j * (2 * np.pi * fc * t + beta * np.sin(2 * np.pi * fm * t)))
    helix_figure(t, s, k, elev, azim)

k10, p10 = player(len(T10), step=3)
ui(sec10, extras=[p10],
   mode=w.ToggleButtons(options=["AM", "FM"], value="AM", description="mode"),
   m=S("AM depth m", 0, 1, 0.05, 0.6, live=False), beta=S("FM index β", 0, 6, 0.25, 3.0, live=False),
   elev=S("elev", 0, 90, 5, 20, live=False), azim=S("azim", -180, 180, 5, -60, live=False), k=k10)

Output()

## 11. A little trickery
Frequency is $+f_0$ in the first half and $-f_0$ in the second half. The **real** side view looks the same on both sides (mirror image). Only **I/Q** shows the direction change: watch the phasor reverse at the centre.

In [15]:
T11 = np.linspace(0, 2, 500)

def sec11(f0, elev, azim, k):
    t = T11
    phase = 2 * np.pi * f0 * (1.0 - np.abs(t - 1.0))      # slope +f0 then -f0
    helix_figure(t, np.exp(1j * phase), k, elev, azim)

k11, p11 = player(len(T11), step=3)
ui(sec11, extras=[p11], f0=S("f0", 1, 4, 0.5, 2.0, live=False),
   elev=S("elev", 0, 90, 5, 20, live=False), azim=S("azim", -180, 180, 5, -60, live=False), k=k11)

Output()

## 12. Pendulum analogy
Use **I** for kinetic energy (velocity) and **Q** for potential energy (position).
Neither alone tells the state (or the energy), but together $I^2+Q^2$ is constant: **this *is* I/Q data.**

In [3]:
T12 = np.linspace(0, 2, 200)

def sec12(theta0, k):
    t = T12
    I = np.cos(2 * np.pi * t)            # velocity (kinetic)
    Q = np.sin(2 * np.pi * t)            # position (potential)
    ang = theta0 * Q
    bx, by = np.sin(ang[k]), -np.cos(ang[k])

    fig, (a, b, c) = plt.subplots(1, 3, figsize=(12, 3.8))
    a.plot([0, bx], [0, by], "k-", lw=2); a.plot([bx], [by], "ro", ms=14)
    a.plot(np.sin(np.linspace(-theta0, theta0, 50)), -np.cos(np.linspace(-theta0, theta0, 50)), color="0.8")
    a.set_xlim(-1.2, 1.2); a.set_ylim(-1.2, 0.2); a.set_aspect("equal"); a.set_title("Pendulum")

    b.plot(I, Q, color="0.8"); b.annotate("", xy=(I[k], Q[k]), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="r", lw=2))
    b.set_xlim(-1.3, 1.3); b.set_ylim(-1.3, 1.3); b.set_aspect("equal")
    b.set_xlabel("I (velocity)"); b.set_ylabel("Q (position)"); b.set_title("State as I/Q")

    c.plot(t, I ** 2, label="kinetic ∝ I²"); c.plot(t, Q ** 2, label="potential ∝ Q²"); c.plot(t, I ** 2 + Q ** 2, "k", label="total")
    c.axvline(t[k], color="r", ls=":"); c.set_ylim(0, 1.2); c.legend(loc="center right", fontsize=8); c.set_xlabel("time"); c.set_title("Energy")
    fig.tight_layout(); plt.show()

k12, p12 = player(len(T12), step=1, interval=50)
ui(sec12, extras=[p12], theta0=S("swing amplitude (rad)", 0.2, 1.2, 0.1, 0.7), k=k12)

Output()

## Exercises
1. Section 1: at which sampling offset does $|I|$ equal the true amplitude? Why can't you know that from real samples alone?
2. Section 5: set the red phase shift to 0. Are the two I curves identical? What differs?
3. Section 7: set $f_1=10,\ f_2=-3$. What changes in the real vs I/Q spectrum?
4. Section 9: set θ = 90°. What happens to I and Q? Does the amplitude change?
5. Section 10: for FM, why is the radius always 1?